In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import importlib 
import time
import pandas as pd 
from Bio import Entrez
import src.fetcher
import src.filters
importlib.reload(src.fetcher)
importlib.reload(src.filters)

from src.fetcher import configure_entrez, fetch_sra_studies, fetch_runs_for_bioproject
from src.filters import check_hard_filters

configure_entrez()

Entrez configured with email: akharya@ucsd.edu


In [3]:
## full pipeline test
ids = fetch_sra_studies('amphibian gut microbiome 16s', max_results = 10)

runs_df = fetch_runs_for_bioproject(ids[0])
filters = check_hard_filters(runs_df)

print('BioProject:', runs_df['BioProject'].iloc[0])
print('Host:', runs_df['ScientificName'].iloc[0])
print('Filters:', filters)

Found 577 studies. Fetching 10...
 Fetched 10 of 10
BioProject: PRJDB37419
Host: gut metagenome
Filters: {'has_16S': True, 'has_WGS': False, 'is_paired': False, 'has_fastq': True, 'passes': False}


In [4]:
# taxonomy search - real host group representation in SRA
import time

tax_groups = {
    "Amphibia":       "txid8292[Organism:exp] AND gut[Text Word]",
    "Reptilia":       "txid1329799[Organism:exp] AND gut[Text Word]",
    "Insecta":        "txid50557[Organism:exp] AND gut[Text Word]",
    "Aves":           "txid8782[Organism:exp] AND gut[Text Word]",
    "Actinopterygii": "txid7898[Organism:exp] AND gut[Text Word]",
    "Mammalia":       "txid40674[Organism:exp] AND gut[Text Word]",
    "Homo sapiens":   "txid9606[Organism:exp] AND gut[Text Word]"
}

print("Host group representation in SRA:\n")

tax_counts = {}
for group, term in tax_groups.items():
    handle = Entrez.esearch(db="sra", term=term, retmax=0)
    record = Entrez.read(handle)
    handle.close()
    count = int(record["Count"])
    tax_counts[group] = count
    print(f"{group}: {count:,} studies")
    time.sleep(0.5)

Host group representation in SRA:

Amphibia: 971 studies
Reptilia: 16,824 studies
Insecta: 18,926 studies
Aves: 16,780 studies
Actinopterygii: 13,921 studies
Mammalia: 256,553 studies
Homo sapiens: 158,506 studies


## Host group representation in SRA (taxonomy-based search, April 2026)

| Host group | Studies | Relative to Mammalia |
|------------|---------|----------------------|
| Amphibia | 971 | 0.4% |
| Actinopterygii | 13,921 | 5.4% |
| Reptilia | 16,824 | 6.6% |
| Aves | 16,780 | 6.5% |
| Insecta | 18,926 | 7.4% |
| Homo sapiens | 158,506 | 61.8% |
| Mammalia | 256,553 | 100% |

### Implications for phylogenetic gap scoring:

Amphibia at 971. Less than 1% of mammalian coverage. This means that amphibians are the highest priority gap for GMToL and therefore deserve the highest phylogenetic gap score. Fish, reptiles, birds and insects are all similarly underrepresented at 5-7% of mammalian coverage. Additional human and mammalian studies and minimal phylogenetic value to GMToL. 

In [6]:
# running hard filters on 20 amphibian gut microbiome studies

amphibian_ids = fetch_sra_studies(
    'txid8292[Organism:exp] AND gut[Text Word]',
    max_results = 20
)

results_summary = {
    "total": 0,
    "passes": 0,
    "fails_no_16S": 0,
    "fails_no_WGS": 0,
    "fails_not_paired": 0,
    "fails_no_fastq": 0
}

for sra_id in amphibian_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is None:
        continue 

    filters = check_hard_filters(runs_df)
    results_summary['total'] +=1

    if filters['passes']:
        results_summary['passes'] +=1
        print(f"PASSES: {runs_df['BioProject'].iloc[0]} | {runs_df['ScientificName'].iloc[0]}")
    else:
        if not filters['has_16S']:
            results_summary['fails_no_16S'] +=1
        if not filters['has_WGS']:
            results_summary['fails_no_WGS'] +=1
        if not filters['is_paired']:
            results_summary['fails_not_paired'] +=1
        if not filters['has_fastq']:
            results_summary['fails_no_fastq'] +=1
        
print("\nSummary of hard filter results:")
print(f"Total studies checked: {results_summary['total']}")
print(f"Passes all filters: {results_summary['passes']}")
print(f"Fails - no 16S: {results_summary['fails_no_16S']}")
print(f"Fails - no WGS: {results_summary['fails_no_WGS']}")
print(f"Fails - not paired: {results_summary['fails_not_paired']}")
print(f"Fails - no FASTQ: {results_summary['fails_no_fastq']}")

Found 971 studies. Fetching 20...
 Fetched 20 of 20

Summary of hard filter results:
Total studies checked: 20
Passes all filters: 0
Fails - no 16S: 0
Fails - no WGS: 20
Fails - not paired: 20
Fails - no FASTQ: 0


All 20 studies fail on the same two filters - no WGS and not paired. 
1) This means that simply searching by host taxonomy might not find paired amphibian studies because they are extremely rare. 
2) smarter strategy: searching for studies that explicity mention being paired and then filter by hosts
